# Exploring Correlations in Python

To calculate correlations between two numeric variables in [pandas](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html#pandas.DataFrame.corr) we can use the `.corr()` method. We can either use a **Pearson**, **Kendall** or **Spearman** method for the calculation. 

- **Pearson** measures linear relationships and is used for continuous data (e.g weight or temperature). 
- **Kendall** and **Spearman** are used for both continuous and ordinal (ranked) data. Both are robust to outliers and non-linearity
- **Spearmann** assumes a [monotonic](https://www.statology.org/monotonic-relationship/) relationship, that is a scenario where a change in one variable is generally associated with a change in a specific direction in another variable.
- **Kendall** makes no assumptions about the data distribution.

For more informations see this [explanation](https://ishanjainoffical.medium.com/choosing-the-right-correlation-pearson-vs-spearman-vs-kendalls-tau-02dc7d7dd01d).

More complex is the calculation of a correlation between nominal (categorical) data. It involves the creation of a **Correspondence Analysis** and the use of the **[Chi-Square Test](https://studyflix.de/statistik/chi-quadrat-test-1683)** to determine if there is a correlation. 

In [10]:
import pandas as pd

## Example: Correlation of Ordinal VOTO Data

Ordinal Data:
- The values represent a meaningful order (0 < 1 < 2 < ... < 10) (e.g. "no trust", "some trust", ... "full trust").
- The differences between values may not be equal (e.g., the difference between 0 and 1 may not be the same as between 8 and 9).
- There is no assumption of equal intervals between the values, which is a key characteristic of ordinal data.

In [ ]:
# open the data
df_raw = pd.read_spss("data/1231_VOTO_CumulativeDataset_Data_projet_v1.0.0.sav")

# define the columns of interest
col_selection = [
    "proposalx", # proposal of voting
    "turnoutx", # result 
    "part", # participated in voting
    "party", # Party that is clearly closest to R's opinion
    "lrsp", # Political left-right placement
    "trust_1", # Extent of trust in federal council
    "trust_2", # Extent of trust in parliament
    "trust_3", # Extent of trust in political parties
    "trust_4", # Extent of trust in the media
    "trust_5", # Extent of trust in the Swiss economy
    "educ" # Highest level of education
]

# create a data subset
df = df_raw[col_selection]
df.head(5)

,proposalx,turnoutx,part,party,lrsp,trust_1,trust_2,trust_3,trust_4,trust_5,educ
0,PI Green Economy,42.998523,did not participate,SVP – Swiss People's Party,extreme left,2,2,don’t know,2,no trust at all,elementary vocational training or apprenticeship
1,PI Green Economy,42.998523,did not participate,SP – Social Democratic Party,2,7,7,7,4,don’t know,maturity or teacher training school
2,PI Green Economy,42.998523,participated,SP – Social Democratic Party,center,no trust at all,9,8,9,9,"post-secondary education, non tertiary"
3,PI Green Economy,42.998523,did not participate,SVP – Swiss People's Party,center,6,no trust at all,2,no trust at all,5,elementary vocational training or apprenticeship
4,PI Green Economy,42.998523,participated,FDP - FDP.The Liberals,center,6,3,4,4,5,elementary vocational training or apprenticeship


In [ ]:
# lets inspect the variables first
unique = df["trust_1"].unique()

# print each variable
for item in unique:
    print(item)

2
7
no trust at all
6
8
complete trust
3
5
9
4
don’t know
1
no answer


### Mapping Ordinal Data to Numeric Values

Problem: The data is a mix of categorical and ordinal data.
- We need to convert the categorical data to ordinal data using the datasets codebook.
- Using the `map()` function of [pandas](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.map.html) we can "find/replace" values in a columns. 

In [ ]:
# create a copy first
df_prep = df.copy()

# map the values to numbers
df_prep["lrsp"] = df_prep["lrsp"].map({
    "extreme left": 0,
    "1": 1,
    "2": 2, 
    "3": 3, 
    "4": 4,
    "center": 5,
    "6": 6,
    "7": 7,
    "8": 8, 
    "9": 9,
    "extreme right": 10
})

df_prep["trust_1"] = df_prep["trust_1"].map({
    "no trust at all": 0,
    "1": 1,
    "2": 2, 
    "3": 3, 
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8, 
    "9": 9,
    "complete trust": 10
})

df_prep["trust_2"] = df_prep["trust_2"].map({
    "no trust at all": 0,
    "1": 1,
    "2": 2, 
    "3": 3, 
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8, 
    "9": 9,
    "complete trust": 10
})

df_prep["trust_3"] = df_prep["trust_3"].map({
    "no trust at all": 0,
    "1": 1,
    "2": 2, 
    "3": 3, 
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8, 
    "9": 9,
    "complete trust": 10
})

df_prep["trust_4"] = df_prep["trust_4"].map({
    "no trust at all": 0,
    "1": 1,
    "2": 2, 
    "3": 3, 
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8, 
    "9": 9,
    "complete trust": 10
})

df_prep["trust_5"] = df_prep["trust_5"].map({
    "no trust at all": 0,
    "1": 1,
    "2": 2, 
    "3": 3, 
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8, 
    "9": 9,
    "complete trust": 10
})

# drop all "don't knows"
df_prep = df_prep.dropna()

df_prep.head()

,proposalx,turnoutx,part,party,lrsp,trust_1,trust_2,trust_3,trust_4,trust_5,educ
2,PI Green Economy,42.998523,participated,SP – Social Democratic Party,5.0,0.0,9.0,8.0,9.0,9.0,"post-secondary education, non tertiary"
3,PI Green Economy,42.998523,did not participate,SVP – Swiss People's Party,5.0,6.0,0.0,2.0,0.0,5.0,elementary vocational training or apprenticeship
4,PI Green Economy,42.998523,participated,FDP - FDP.The Liberals,5.0,6.0,3.0,4.0,4.0,5.0,elementary vocational training or apprenticeship
6,PI Green Economy,42.998523,did not participate,FDP - FDP.The Liberals,5.0,7.0,7.0,7.0,7.0,7.0,elementary vocational training or apprenticeship
7,PI Green Economy,42.998523,did not participate,does not affiliate with a party,5.0,7.0,8.0,4.0,5.0,8.0,compulsory school


### Using the `corr()` method

The `corr()` method can be attached to a dataframe and returns a correlation matrix. It takes a `method` parameter that specifies the method to use for calculating the correlation. The default method is `pearson`.

In [14]:
# Spearman correlation for ordinal (ranked) data
cm_kendall = df_prep[['trust_1', 'trust_2']].corr(method='kendall')
print(cm_kendall)

          trust_1   trust_2
trust_1  1.000000  0.600572
trust_2  0.600572  1.000000


In [17]:
cm_spearman = df_prep[['trust_1', 'trust_2']].corr(method='spearman')
print(cm_spearman)

          trust_1   trust_2
trust_1  1.000000  0.693222
trust_2  0.693222  1.000000


Just add more variables to the `corr` method to see the correlation between all variables.

In [19]:
cm = df_prep[['lrsp', 'trust_1', 'trust_2', 'trust_3', 'trust_4', 'trust_5']].corr(method='kendall')
cm

,lrsp,trust_1,trust_2,trust_3,trust_4,trust_5
lrsp,1.000000,-0.003815,0.025010,0.066870,-0.080198,0.175026
trust_1,-0.003815,1.000000,0.600572,0.334595,0.261120,0.332916
trust_2,0.025010,0.600572,1.000000,0.385727,0.248298,0.357661
trust_3,0.066870,0.334595,0.385727,1.000000,0.290953,0.276844
trust_4,-0.080198,0.261120,0.248298,0.290953,1.000000,0.189374
trust_5,0.175026,0.332916,0.357661,0.276844,0.189374,1.000000


## Styling Pandas Dataframes

For more information see the [official documentation](https://pandas.pydata.org/docs/user_guide/style.html) and [this](https://towardsdatascience.com/how-to-style-pandas-dataframes-like-a-pro-541c84142c17/) blogpost.

In [27]:
# this returns a styled table
styled_table = cm.style.background_gradient(cmap="RdYlGn", vmin=-1, vmax=1).format(precision=2)

# the styled tables must be displayed using the display command
display(styled_table)

,lrsp,trust_1,trust_2,trust_3,trust_4,trust_5
lrsp,1.00,-0.00,0.03,0.07,-0.08,0.18
trust_1,-0.00,1.00,0.60,0.33,0.26,0.33
trust_2,0.03,0.60,1.00,0.39,0.25,0.36
trust_3,0.07,0.33,0.39,1.00,0.29,0.28
trust_4,-0.08,0.26,0.25,0.29,1.00,0.19
trust_5,0.18,0.33,0.36,0.28,0.19,1.00


### Exporting a styled table to excel

In [ ]:
styled_table.to_excel('styled_table.xlsx', index=False)

## Bonus: Machine learning models to make predictions from correlations

The Python library **[scikit-learn](https://scikit-learn.org/stable/index.html)** provides a wide range of machine learning models that can be used to predict data from strong correlations.
The library offers [linear regression methods](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) for continous linear data or [decision trees](https://scikit-learn.org/stable/modules/tree.html) for ordinal data. 